# Sprint 01 — US 1.1 : Enquête sur les données

Objectif : explorer les CSV bruts, extraire les premières métriques (lignes, colonnes, doublons, valeurs manquantes) et repérer les enjeux d'anonymisation / de biais.

In [ ]:
from indusense.data import loaders

telemetry = loaders.load_telemetry()
incidents = loaders.load_incidents()
telemetry.shape, incidents.shape

In [ ]:
telemetry.head()

In [ ]:
telemetry.info()
telemetry.isna().sum()

## US 1.1 (suite) — Pipeline d'ingestion & analyse des incidents

Au-delà de l'exploration manuelle, on **industrialise** le traitement des relevés
d'incidents via un pipeline reproductible et tracé (`indusense-incidents`).
À chaque exécution, il produit dans `artifacts/ingestions/incidents/AAAAMMJJHHMM/` :

- un **dataset anonymisé + enrichi** (Parquet, versionné par horodatage) ;
- **8 graphes SVG** annotés (distributions, histogrammes par signal/machine, corrélations) ;
- un **journal de runs** (`runs.json` / `runs.md`) et une note **`METHODOLOGIE.md`**.

Les cellules ci-dessous rejouent les étapes clés (cf. `src/indusense/data/incidents_ingest.py`).

In [ ]:
from indusense.data.incidents_ingest import load_incidents_raw

incidents_raw = load_incidents_raw()
print("dimensions :", incidents_raw.shape)
incidents_raw.isna().sum()

### 1. Anonymisation des opérateurs (minimisation RGPD)

`operator_name` et `operator_badge` sont des identifiants directs (DCP), **non
nécessaires** à l'analyse machine → **supprimés** (anonymisation irréversible).
`comment` est conservé (saisie guidée non-DCP). Justification complète :
`artifacts/ingestions/incidents/METHODOLOGIE.md`.

In [ ]:
from indusense.data.anonymize import anonymize_operators

incidents_anon = anonymize_operators(incidents_raw)
print("colonnes supprimées :", set(incidents_raw.columns) - set(incidents_anon.columns))
print("dimensions après anonymisation :", incidents_anon.shape)

### 2. Enrichissement & indice de confiance du signalement

On dérive le **signal dominant**, des axes temporels (jour / semaine ISO / weekday)
et un **indice de confiance par incident** (cohérence des flags + présence du
commentaire + validité machine/sévérité), borné dans [0, 1].

In [ ]:
from indusense.data.incidents_ingest import compute_confidence, enrich

incidents = compute_confidence(enrich(incidents_anon))
display(incidents["signal"].value_counts())
incidents["confidence"].describe()

### 3. Exécution du pipeline complet (dataset versionné + graphes + journal)

`run_ingestion()` enchaîne chargement → anonymisation → enrichissement → confiance,
puis écrit le Parquet, génère les 8 SVG et met à jour le journal + la méthodologie.

In [ ]:
from indusense.data import incidents_ingest as ing

meta = ing.run_ingestion()
print(f"Run {meta['run_id']}")
print(
    f"  lignes={meta['n_lignes']} colonnes={meta['n_colonnes']} "
    f"machines={meta['machines_uniques']} NaN={meta['n_nan_total']} "
    f"confiance_moy={meta['confidence_moyenne']}"
)
print("  figures :", ", ".join(meta["figures"]))

### 4. Visualisation des graphes (les 8 figures du dernier run)

Chaque SVG embarque son titre et une légende explicative. Trois familles :

- **Distributions** — incidents par jour, par semaine (ISO) et par shift.
- **Histogrammes** — par signal et par machine (barres = comptes, courbe = confiance
  moyenne), plus la distribution de l'indice de confiance.
- **Corrélations** — *Pearson* (linéaire, sensible aux extrêmes) et *Spearman*
  (monotone, sur les rangs, robuste aux outliers) ; les comparer distingue une vraie
  relation d'un artefact (cf. `METHODOLOGIE.md`).

In [ ]:
from IPython.display import SVG, display

from indusense import config

run_dir = sorted(p for p in config.INGEST_INCIDENTS_DIR.glob("2*") if p.is_dir())[-1]
figures = [
    # Distributions
    "dist_incidents_par_jour.svg",
    "dist_incidents_par_semaine.svg",
    "dist_incidents_par_shift.svg",
    # Histogrammes
    "hist_par_signal.svg",
    "hist_par_machine.svg",
    "hist_confiance.svg",
    # Corrélations
    "correlation_signaux_pearson.svg",
    "correlation_signaux_spearman.svg",
]
for name in figures:
    display(SVG(filename=str(run_dir / "figures" / name)))

### Livrables produits

| Livrable | Emplacement |
|---|---|
| Dataset anonymisé + enrichi (Parquet) | `artifacts/ingestions/incidents/<run>/incidents_anonymized.parquet` |
| 8 graphes SVG annotés | `artifacts/ingestions/incidents/<run>/figures/` |
| Métadonnées du run | `artifacts/ingestions/incidents/<run>/run_metadata.json` |
| Journal des runs | `artifacts/ingestions/incidents/runs.json` · `runs.md` |
| Méthodologie (RGPD + indice de confiance) | `artifacts/ingestions/incidents/METHODOLOGIE.md` |

▶️ Reproductible en une commande : `uv run indusense-incidents`.